
# Predicting Air Quality Using Traffic Data  
**Random Forest Regression**

This notebook builds a predictive model to estimate daily average NO₂ levels
using traffic count data.  
It follows these steps:
1. Load and clean air quality data  
2. Load and reshape traffic data  
3. Merge datasets by date and station  
4. Train a Random Forest regression model  
5. Evaluate model performance  

You can run this notebook cell-by-cell once you place the datasets
in the expected folders.


## 1. Load and Prepare Air Quality Data

In [ ]:

import pandas as pd
import glob
import re
import numpy as np

# Read air quality data
all_data = []
for file_path in glob.glob("Air Quality Dataset/aq_data/*.csv"):
    with open(file_path, 'r') as f:
        meta_lines = [next(f) for _ in range(10)]
        station_name = meta_lines[1].split(',')[1].strip()
        lat = meta_lines[3].split(',')[1].strip()
        lon = meta_lines[4].split(',')[1].strip()
    
    df = pd.read_csv(file_path, skiprows=10, index_col=False)
    df['Station_Name'] = station_name
    df['Latitude'] = lat
    df['Longitude'] = lon
    all_data.append(df)

aq_df = pd.concat(all_data, ignore_index=True)

# Replace missing value codes with NA
aq_df.replace([9999, -999], pd.NA, inplace=True)

# Calculate daily average NO2 from hourly columns
hourly_cols = [f'H{i:02d}' for i in range(1, 25)]
aq_df['daily_total'] = aq_df[hourly_cols].mean(axis=1)

# Clean dates
aq_df['Date'] = pd.to_datetime(aq_df['Date'], errors='coerce')
aq_df = aq_df.dropna(subset=['Date'])
aq_df = aq_df[aq_df['daily_total'] >= 0]

print(f"Air Quality data loaded: {len(aq_df)} records")


## 2. Load and Prepare Traffic Data

In [ ]:

traffic_path = "Traffic Dataset - Statistics Canada/filtered_roads_with_sums.csv"
traffic_df = pd.read_csv(traffic_path)

# Identify date columns (xYYYY_MM_DD)
date_col_pattern = re.compile(r"^x\d{4}_\d{2}_\d{2}$")
traffic_date_cols = [c for c in traffic_df.columns if date_col_pattern.match(str(c))]

# Convert to long format
traffic_long = []
for _, row in traffic_df.iterrows():
    camera_road = row['camera_road']
    for col in traffic_date_cols:
        if pd.notna(row[col]):
            date = pd.to_datetime(col[1:], format='%Y_%m_%d')
            traffic_long.append({
                'Date': date,
                'camera_road': camera_road,
                'traffic_count': row[col]
            })

traffic_long_df = pd.DataFrame(traffic_long)
print(f"Traffic data loaded: {len(traffic_long_df)} records")


## 3. Map Traffic Locations to Air Quality Stations

In [ ]:

location_mapping = {
    'LAWRENCE AVE E / KENNEDY RD': 'Toronto East (33003)',
    'STEELES AVE W / DUFFERIN ST': 'Toronto North (34021)',
    'FRONT ST W / JOHN ST / PRIVATE ACCESS': 'Toronto Downtown (31129)',
    'ISLINGTON AVE / 401 C W ISLINGTON N RAMP / ALLENBY AVE': 'Toronto West (35125)'
}

traffic_long_df['Station_Name'] = traffic_long_df['camera_road'].map(location_mapping)


## 4. Merge Air Quality and Traffic Data

In [ ]:

# Select station
selected_station = 'Toronto North (34021)'

# Filter datasets
aq_station = aq_df[aq_df['Station_Name'] == selected_station][['Date', 'daily_total']]
traffic_station = traffic_long_df[traffic_long_df['Station_Name'] == selected_station][['Date', 'traffic_count']]

# Merge on Date
merged_data = pd.merge(aq_station, traffic_station, on='Date', how='inner')
merged_data = merged_data.sort_values('Date').reset_index(drop=True)

print(f"Merged records: {len(merged_data)}")
merged_data.head()


## 5. Train a Random Forest Model

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Features and target
X = merged_data[['traffic_count']]
y = merged_data['daily_total']

# Train-test split (time-agnostic for simplicity)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Random Forest model
rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

# Predictions
y_pred = rf.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error: {mae:.3f}")
print(f"R² Score: {r2:.3f}")



## 6. Interpretation

- **MAE** indicates the average prediction error in NO₂ units  
- **R²** shows how much variance in air quality is explained by traffic volume  

You can extend this model by:
- Adding lagged traffic variables (e.g., previous-day traffic)
- Including weather data
- Training separate models for each station
